In [1]:

# Imports

import os
import cv2
import numpy as np
import pandas as pd
import pydicom
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:

# Paths (Adjust if needed)


DICOM_DIR = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"
ANNOTATION_CSV = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv"

OUTPUT_ROOT = "/kaggle/working/data"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:

#  Load Annotations


df = pd.read_csv(ANNOTATION_CSV)

image_ids = df["image_id"].unique()

print("Total images:", len(image_ids))

Total images: 15000


In [ ]:

# Train / Val / Test Split


train_ids, temp_ids = train_test_split(image_ids, test_size=0.2, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

split_map = {}
for i in train_ids: split_map[i] = "train"
for i in val_ids: split_map[i] = "val"
for i in test_ids: split_map[i] = "test"

In [ ]:

# Create Folder Structure


for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_ROOT}/labels/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_ROOT}/classification/{split}/normal", exist_ok=True)
    os.makedirs(f"{OUTPUT_ROOT}/classification/{split}/abnormal", exist_ok=True)

In [ ]:

# Image Preprocessing


def preprocess_image(dicom_path, target_size=640):

    ds = pydicom.dcmread(dicom_path)
    img = ds.pixel_array.astype(np.float32)

    # Save original size BEFORE resizing
    orig_h, orig_w = img.shape[:2]

    # Min-max normalization
    img = (img - img.min()) / (img.max() - img.min() + 1e-6)
    img = (img * 255).astype(np.uint8)

    # Resize first
    img = cv2.resize(img, (target_size, target_size))

    # CLAHE (after resize)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # Light denoising
    img = cv2.GaussianBlur(img, (3,3), 0)

    return img, orig_h, orig_w

In [ ]:

# Main Processing Loop


TARGET_SIZE = 640

for image_id in tqdm(image_ids):

    dicom_path = os.path.join(DICOM_DIR, image_id + ".dicom")
    if not os.path.exists(dicom_path):
        continue

    img, orig_h, orig_w = preprocess_image(dicom_path, TARGET_SIZE)

    split = split_map[image_id]

    # Save processed image
    img_path = f"{OUTPUT_ROOT}/images/{split}/{image_id}.png"
    cv2.imwrite(img_path, img)

    # Get annotation rows
    rows = df[df["image_id"] == image_id]

    label_lines = []
    is_abnormal = False

    for _, row in rows.iterrows():

        # Skip normal label rows
        if row["class_name"] == "No finding":
            continue

        # If bbox missing, skip
        if pd.isna(row["x_min"]):
            continue

        is_abnormal = True

        x_min = row["x_min"]
        y_min = row["y_min"]
        x_max = row["x_max"]
        y_max = row["y_max"]

        # Convert to YOLO normalized format (binary class = 0)
        x_center = ((x_min + x_max) / 2) / orig_w
        y_center = ((y_min + y_max) / 2) / orig_h
        box_w = (x_max - x_min) / orig_w
        box_h = (y_max - y_min) / orig_h

        # Safety check (prevent corrupt labels)
        if 0 < x_center < 1 and 0 < y_center < 1 and 0 < box_w < 1 and 0 < box_h < 1:
            label_lines.append(f"0 {x_center} {y_center} {box_w} {box_h}")

    # Save detection label ONLY if abnormal
    if is_abnormal and len(label_lines) > 0:
        with open(f"{OUTPUT_ROOT}/labels/{split}/{image_id}.txt", "w") as f:
            f.write("\n".join(label_lines))

        cv2.imwrite(f"{OUTPUT_ROOT}/classification/{split}/abnormal/{image_id}.png", img)

    else:
        # Normal image (no label file for YOLO)
        cv2.imwrite(f"{OUTPUT_ROOT}/classification/{split}/normal/{image_id}.png", img)

100%|██████████| 15000/15000 [4:42:44<00:00,  1.13s/it]


In [ ]:

#  Create YOLO dataset.yaml


yaml_content = f"""
path: {OUTPUT_ROOT}
train: images/train
val: images/val

nc: 1
names: ['abnormal']
"""

with open(f"{OUTPUT_ROOT}/dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Binary preprocessing complete.")

Binary preprocessing complete.


In [ ]:
=
# Zip Entire data Folder


import shutil
import os

zip_path = "/kaggle/working/vindr_binary_preprocessed.zip"

# This ensures the folder "data" itself appears inside the zip
shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=os.path.dirname(OUTPUT_ROOT),
    base_dir=os.path.basename(OUTPUT_ROOT)
)

print("Dataset zipped at:", zip_path)

Dataset zipped at: /kaggle/working/vindr_binary_preprocessed.zip
